In [1]:
%uv pip install datasets

Using Python 3.12.6 environment at: /usr/local
Resolved 36 packages in 238ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
dill       ------------------------------ 32.00 KiB/116.86 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 48.00 KiB/116.86 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 48.00 KiB/116.86 KiB
xxhash     ------------------------------     0 B/189.34 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 48.00 KiB/116.86 KiB
xxhash     ------------------------------ 16.00 KiB/189.34 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 48.00 KiB/116.86 KiB
xxhash     ------------------------------ 16.00 KiB/189.34 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 48.00 KiB/116.86 KiB
multiprocess ------------------------------     0 B/146.76 KiB
xxhash     --------------

In [2]:
#importing the libraries that we need
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from datasets import load_dataset
from torch.utils.data import DataLoader, default_collate
import numpy as np
import random
import os
import time
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device") #Checking whether we're on a CPU or GPU runtime


Using cuda:0 device


In [3]:
#importing the EYEPACS dataset from Hugging Face and creating the 70-30 train test split
ds = load_dataset("bumbledeep/eyepacs")
data = ds['train']
split = data.train_test_split(test_size=0.3, seed=42)
image_datasets = {'train': split['train'], 'test': split['test']}
print("Train size:", len(image_datasets['train']), "Test size:", len(image_datasets['test']))


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/467M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/train-00003-of-00014.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

...

data/train-00011-of-00014.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00012-of-00014.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/train-00013-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/35108 [00:00<?, ? examples/s]

Train size: 24575 Test size: 10533


In [23]:
#implementing transformations to help the model detect and learn subtle features more easily
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

data_transforms = {}
data_transforms['train'] = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02,0.2))
])
data_transforms['test'] = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

#turn images from the dataset (received as batches) into tensors that the model can understand
def collate_fn(batch, phase='train'):
    imgs = []
    labels = []
    for ex in batch:
        img = ex.get('image', None)
        if isinstance(img, list):
            img = img[0]
        tensor = data_transforms[phase](img.convert("RGB"))
        imgs.append(tensor)
        labels.append(int(ex.get('label_code', ex.get('labels', ex.get('label', 0)))))
    px = torch.stack(imgs, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)
    return px, labels
def collate_train(batch):
    return collate_fn(batch, phase='train')
def collate_test(batch):
    return collate_fn(batch, phase='test')


In [24]:
#dataloaders to turn the dataset into minibatches that will be read by the model.
batch_size = 8
num_workers = 2
pin_memory = torch.cuda.is_available()
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_train, drop_last=True),
    'test':  DataLoader(image_datasets['test'],  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_test, drop_last=False)
}
print("Dataloaders built. Train size:", len(image_datasets['train']), "Test size:", len(image_datasets['test']), "batch_size:", batch_size)


Dataloaders built. Train size: 24575 Test size: 10533 batch_size: 8


In [25]:
%uv pip install -q scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [26]:
#calculate a weight for every class. The model will pay more attention to classes with higher weights 
#than to lower weighted ones, to compensate for class imbalance in our dataset.
from sklearn.utils.class_weight import compute_class_weight
num_classes = 5
y_labels = []
stream_train = load_dataset("bumbledeep/eyepacs", split="train", streaming=False)
for ex in stream_train:
    y_labels.append(int(ex['label_code']))
classes = np.arange(num_classes)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(y_labels))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("class counts (approx):", np.bincount(y_labels, minlength=num_classes), "class weights:", class_weights)


class counts (approx): [25802  2438  5288   872   708] class weights: [0.27213394 2.88006563 1.32783661 8.05229358 9.91751412]


In [28]:
#Load a pretrained Resnet 101 model
model = models.resnet101(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
#Unfreeze layers 3 and 4 of the model, as well as the final layer, so they can train and learn new features
#specific to our dataset
for p in model.parameters(): p.requires_grad = False
for name, module in model.named_children():
    if name in ('layer3','layer4'):
        for p in module.parameters(): p.requires_grad = True
for p in model.fc.parameters(): p.requires_grad = True
#Creating two parameter groups that will train at different learning rates. This is a best practice for 
#transfer learning.
backbone_params = [p for n,p in model.named_parameters() if p.requires_grad and not n.startswith('fc')]
head_params = [p for n,p in model.named_parameters() if p.requires_grad and (n.startswith('fc') or n.startswith('fc.'))]
optimizer = optim.Adam([{'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-4},{'params': head_params, 'lr': 1e-4, 'weight_decay': 1e-4}])


In [29]:
#Create a loss function that uses the class weights to handle imbalanced data
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print("Using CrossEntropyLoss with class weights on device:", class_weights_tensor.device)


Using CrossEntropyLoss with class weights on device: cuda:0


In [30]:
import copy
def train(model, dataloaders, criterion, optimizer, epochs, device):
    model.to(device)
    best_acc = 0.0
    if hasattr(criterion, 'weight') and criterion.weight is not None:
        criterion.weight = criterion.weight.to(device)
    #Save the best weights seen so far
    best_model_wts = copy.deepcopy(model.state_dict())
    start_epoch = 0
    for epoch in range(start_epoch, epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        #Two phases for the epoch: train and test
        for phase in ('train', 'test'):
            if phase == 'train':
                model.train() #do the standard training for the model in this phase
            else:
                model.eval() #find the loss but don't backpropagate like in training
            epoch_start = time.time()
            running_loss = 0.0
            running_corrects = 0
            samples = 0
            for inputs, labels in dataloaders[phase]: #iterate over the minibatches we created
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True).long()
                if phase == 'train':
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                bs = inputs.size(0)
                running_loss += loss.item() * bs
                running_corrects += (preds == labels).sum().item()
                samples += bs
            epoch_loss = running_loss / samples
            epoch_acc = running_corrects / samples
            if phase == 'train':
                print(f" train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
            else:
                print(f" val   Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict()) 
                    print(f" New best val acc {best_acc:.4f}")
    model.load_state_dict(best_model_wts) #saving the best weights into the model
    return model

In [28]:
#Create a WeightedRandomSampler so that samples from rare classes are processed more often
from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np
train_labels = np.array([int(x['label_code']) for x in image_datasets['train']])
class_count = np.bincount(train_labels, minlength=num_classes)
w = 1.0 / (class_count + 1e-12)
w[1] = w[1] * 2
sample_weights = w[train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=False)
dataloaders['train'] = DataLoader(image_datasets['train'], batch_size=8, sampler=sampler, num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_train)

In [12]:
#Train the model
model = train(model, dataloaders, criterion, optimizer, epochs=2, device=device)
print("Training finished")



Epoch 1/2
 train Loss: 0.6698 Acc: 0.6788
 val   Loss: 5.6821 Acc: 0.7301
 New best val acc 0.7301

Epoch 2/2
 train Loss: 0.6378 Acc: 0.7083
 val   Loss: 6.0731 Acc: 0.7301
Training finished


In [21]:
criterion_backup = criterion
criterion = nn.CrossEntropyLoss()   

dataloaders_backup = dataloaders
dataloaders['train'] = DataLoader(image_datasets['train'],
                                  batch_size=16, shuffle=True,
                                  num_workers=2, pin_memory=pin_memory,
                                  collate_fn=collate_train, drop_last=True)

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
for p in model.parameters(): p.requires_grad = False
for p in model.fc.parameters(): p.requires_grad = True
opt_tmp = optim.Adam(model.fc.parameters(), lr=1e-3)

model = train(model, dataloaders, criterion, opt_tmp, epochs=3, device=device)


Epoch 1/3
 train Loss: 0.7928 Acc: 0.7445
 val   Loss: 0.8145 Acc: 0.7593
 New best val acc 0.7593

Epoch 2/3
 train Loss: 0.7856 Acc: 0.7489
 val   Loss: 0.7676 Acc: 0.7512

Epoch 3/3
 train Loss: 0.7824 Acc: 0.7450
 val   Loss: 0.7518 Acc: 0.7551


In [29]:
model = train(model, dataloaders, criterion, optimizer, epochs=2, device=device)



Epoch 1/2
 train Loss: 0.8516 Acc: 0.7392
 val   Loss: 0.8437 Acc: 0.7539
 New best val acc 0.7539

Epoch 2/2
 train Loss: 0.8515 Acc: 0.7375
 val   Loss: 0.8568 Acc: 0.7536


In [32]:
#Collecting predictions and probabilities for each class, to be used for evaluation scores
test_loader = DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_test)
model.to(device)
model.eval()
y_true = []
y_pred = []
probs_list = []
max_conf_list = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(inputs)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_list.append(probs.cpu().numpy())
        max_conf_list.extend(probs.max(dim=1)[0].cpu().numpy().tolist())
probs_np = np.concatenate(probs_list, axis=0)
y_true_np = np.array(y_true)
y_pred_np = np.array(y_pred)
max_conf_np = np.array(max_conf_list)
print("Predictions samples:", len(y_true_np), "probs shape:", probs_np.shape)


Predictions samples: 10533 probs shape: (10533, 5)


In [33]:
#calculating evaluation metrics - precision, recall, F1 scores, confusion matrix, and accuracy
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
prec_per_class = precision_score(y_true_np, y_pred_np, average=None, zero_division=0)
for cls_idx, prec in enumerate(prec_per_class):
    print(f"Class {cls_idx} precision: {prec:.4f}")
print("Macro precision:", precision_score(y_true_np, y_pred_np, average='macro', zero_division=0))
print("Micro precision:", precision_score(y_true_np, y_pred_np, average='micro', zero_division=0))
rec_per_class = recall_score(y_true_np, y_pred_np, average=None, zero_division=0)
for cls_idx, rec in enumerate(rec_per_class):
    print(f"Class {cls_idx} recall: {rec:.4f}")
print("Macro recall:", recall_score(y_true_np, y_pred_np, average='macro', zero_division=0))
print("Micro recall:", recall_score(y_true_np, y_pred_np, average='micro', zero_division=0))
f1_per_class = f1_score(y_true_np, y_pred_np, average=None, zero_division=0)
for cls_idx, f1 in enumerate(f1_per_class):
    print(f"Class {cls_idx} F1: {f1:.4f}")
print("Macro F1:", f1_score(y_true_np, y_pred_np, average='macro', zero_division=0))
print("Micro F1:", f1_score(y_true_np, y_pred_np, average='micro', zero_division=0))
cm = confusion_matrix(y_true_np, y_pred_np)
print("Confusion matrix (rows=true, cols=pred):")
print(cm)
pred_counts = np.bincount(y_pred_np, minlength=probs_np.shape[1])
print("Predicted counts per class:")
for c, cnt in enumerate(pred_counts):
    print(f" Predicted as class {c}: {cnt}")
num_classes = probs_np.shape[1]
for c in range(num_classes):
    idx = (y_true_np == c)
    if idx.sum() == 0:
        acc = float('nan')
    else:
        acc = (y_pred_np[idx] == y_true_np[idx]).sum() / idx.sum()
    print(f"Class {c} accuracy: {acc if not np.isnan(acc) else 'N/A'}")
overall_acc = (y_pred_np == y_true_np).sum() / len(y_true_np)
print("Overall accuracy:", overall_acc)


Class 0 precision: 0.7944
Class 1 precision: 0.0000
Class 2 precision: 0.4906
Class 3 precision: 0.3149
Class 4 precision: 0.8000
Macro precision: 0.4799827413210858
Micro precision: 0.7539162631728852
Class 0 recall: 0.9694
Class 1 recall: 0.0000
Class 2 recall: 0.2116
Class 3 recall: 0.5379
Class 4 recall: 0.0180
Macro recall: 0.34738240006240134
Micro recall: 0.7539162631728852
Class 0 F1: 0.8733
Class 1 F1: 0.0000
Class 2 F1: 0.2957
Class 3 F1: 0.3972
Class 4 F1: 0.0352
Macro F1: 0.32027096929926857
Micro F1: 0.7539162631728852
Confusion matrix (rows=true, cols=pred):
[[7455    0  193   41    1]
 [ 708    0   32   10    0]
 [1070    0  340  197    0]
 [  63    0   59  142    0]
 [  88    0   69   61    4]]
Predicted counts per class:
 Predicted as class 0: 9384
 Predicted as class 1: 0
 Predicted as class 2: 693
 Predicted as class 3: 451
 Predicted as class 4: 5
Class 0 accuracy: 0.9694408322496749
Class 1 accuracy: 0.0
Class 2 accuracy: 0.21157436216552583
Class 3 accuracy: 0.537

In [36]:
#Calculating the time taken to run an epoch on average
import time, math, copy, json
import numpy as np
import torch

epochs_to_estimate = 1
max_batches = 10
warmup_batches = 1

if 'model' not in globals() or 'dataloaders' not in globals() or 'criterion' not in globals():
    raise RuntimeError("This needs `model`, `dataloaders`, and `criterion` defined in the notebook.")

train_loader = dataloaders['train']
train_n = len(image_datasets['train']) if 'image_datasets' in globals() else None
train_bs = getattr(train_loader, 'batch_size', None)
if train_bs is None:
    train_bs = train_loader.batch_sampler.batch_size

if train_bs is not None and train_n is not None:
    batches_per_epoch = math.ceil(train_n / train_bs)
else:
    batches_per_epoch = len(train_loader)

device = next(model.parameters()).device
print(f"Device: {device}. Estimated batches/epoch: {batches_per_epoch}. Timing up to {max_batches or 'full epoch'} batch(es).")

saved_state = copy.deepcopy(model.state_dict())
model.to(device)

it = iter(train_loader)
for i in range(warmup_batches):
    inp, lab = next(it)
    inp = inp.to(device, non_blocking=True)
    lab = lab.to(device, non_blocking=True)
    model.train()
    model.zero_grad(set_to_none=True)
    out = model(inp)
    loss = criterion(out, lab)
    loss.backward()
    model.zero_grad(set_to_none=True)

batch_times = []
while True:
    if max_batches is not None and len(batch_times) >= max_batches:
        break
    inp, lab = next(it)
    inp = inp.to(device, non_blocking=True)
    lab = lab.to(device, non_blocking=True)
    model.train()
    model.zero_grad(set_to_none=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    out = model(inp)
    loss = criterion(out, lab)
    loss.backward()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.time()
    batch_times.append(t1 - t0)
    model.zero_grad(set_to_none=True)

if len(batch_times) == 0:
    raise RuntimeError("No batches timed. Check the train dataloader.")

avg_batch = float(np.mean(batch_times))
std_batch = float(np.std(batch_times))
print(f"Timed {len(batch_times)} batches, average = {avg_batch:.4f}s")

if batches_per_epoch is not None:
    per_epoch = batches_per_epoch * avg_batch
    total = epochs_to_estimate * per_epoch
    print(f"1 epoch time: {per_epoch:.1f}s ({time.strftime('%H:%M:%S', time.gmtime(int(round(per_epoch))))})")
else:
    per_epoch = None
    total = None
    print("Could not determine batches per epoch automatically. Showing per-batch average only.")

model.load_state_dict(saved_state)
model.zero_grad(set_to_none=True)
summary = {
    "device": str(device),
    "sampled_batches": len(batch_times),
    "avg_batch_time_s": avg_batch,
    "std_batch_time_s": std_batch,
    "batches_per_epoch_est": batches_per_epoch,
    "per_epoch_est_s": per_epoch,
    "total_est_s": total
}
print(json.dumps(summary, indent=2))

Device: cuda:0. Estimated batches/epoch: 3072. Timing up to 10 batch(es).
Timed 10 batches, average = 0.0420s
1 epoch time: 129.1s (00:02:09)
{
  "device": "cuda:0",
  "sampled_batches": 10,
  "avg_batch_time_s": 0.042018795013427736,
  "std_batch_time_s": 0.0024331009126633673,
  "batches_per_epoch_est": 3072,
  "per_epoch_est_s": 129.08173828125,
  "total_est_s": 129.08173828125
}


In [31]:
#Finding the number of samples(images) for each class in the dataset
from collections import Counter

def get_class_counts(dataset, label_keys=('label_code', 'label', 'labels')):
    """
    dataset: HF dataset object or any iterable of examples where each example is a dict.
    label_keys: possible keys holding the label.
    Returns a Counter with counts per class (int).
    """
    counts = Counter()
    for ex in dataset:
        val = None
        for k in label_keys:
            if k in ex and ex[k] is not None:
                val = ex[k]
                break
        if val is None:
            continue
        if isinstance(val, list):
            val = val[0]
        counts[int(val)] += 1
    return counts

train_counts = get_class_counts(image_datasets['train'])
test_counts  = get_class_counts(image_datasets['test'])

print("Train class counts:", dict(sorted(train_counts.items())))
print("Test  class counts:", dict(sorted(test_counts.items())))

from copy import deepcopy
total_counts = deepcopy(train_counts)
for k,v in test_counts.items():
    total_counts[k] += v
print("Total class counts (train+test):", dict(sorted(total_counts.items())))

Train class counts: {0: 18112, 1: 1688, 2: 3681, 3: 608, 4: 486}
Test  class counts: {0: 7690, 1: 750, 2: 1607, 3: 264, 4: 222}
Total class counts (train+test): {0: 25802, 1: 2438, 2: 5288, 3: 872, 4: 708}
